In [1]:
import pandas as pd
import numpy as np

revenue = pd.read_csv("revenue.csv")
expenses = pd.read_csv("expenses.csv")
budget = pd.read_csv("budget.csv")
kpi = pd.read_csv("kpi_metrics.csv")
scenarios = pd.read_csv("scenario_inputs.csv")

revenue["Month"] = pd.to_datetime(revenue["Month"])
expenses["Month"] = pd.to_datetime(expenses["Month"])
budget["Month"] = pd.to_datetime(budget["Month"])
kpi["Month"] = pd.to_datetime(kpi["Month"])

print("Files loaded successfully.")

Files loaded successfully.


/tmp/ipykernel_8111/2545670773.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  revenue["Month"] = pd.to_datetime(revenue["Month"])
/tmp/ipykernel_8111/2545670773.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  expenses["Month"] = pd.to_datetime(expenses["Month"])
/tmp/ipykernel_8111/2545670773.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  budget["Month"] = pd.to_datetime(budget["Month"])
/tmp/ipykernel_8111/2545670773.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is con

In [3]:
monthly_revenue = revenue.groupby("Month", as_index=False).agg({
    "Subscription_Revenue": "sum",
    "Transaction_Fee_Revenue": "sum",
    "Premium_Analytics_Revenue": "sum",
    "Setup_Fee_Revenue": "sum",
    "Total_Revenue": "sum"
})

monthly_revenue

,Month,Subscription_Revenue,Transaction_Fee_Revenue,Premium_Analytics_Revenue,Setup_Fee_Revenue,Total_Revenue
0,2024-01-01,1098,1690,199,1200,4187
1,2024-01-02,1098,1755,199,0,3052
2,2024-01-03,1397,2262,199,500,4358
3,2024-01-04,2196,3289,398,700,6583
4,2024-01-05,2495,3887,398,500,7280
5,2024-01-06,3294,4940,597,700,9531


In [4]:
expense_variance = expenses.copy()

expense_variance["Expense_Variance"] = expense_variance["Actual_Expense"] - expense_variance["Budget_Expense"]
expense_variance["Expense_Variance_%"] = expense_variance["Expense_Variance"] / expense_variance["Budget_Expense"]

expense_variance.sort_values("Expense_Variance", ascending=False).head(10)

,Month,Department,Expense_Category,Budget_Expense,Actual_Expense,Vendor,Cost_Type,Expense_Variance,Expense_Variance_%
25,2024-01-05,Marketing,Marketing Spend,10000,11600,Google Ads,Variable,1600,0.160000
13,2024-01-03,Marketing,Marketing Spend,9000,10300,Google Ads,Variable,1300,0.144444
1,2024-01-01,Marketing,Marketing Spend,8000,9200,Google Ads,Variable,1200,0.150000
32,2024-01-06,Product,Cloud Hosting,5800,7000,AWS,Variable,1200,0.206897
26,2024-01-05,Product,Cloud Hosting,5500,6500,AWS,Variable,1000,0.181818
20,2024-01-04,Product,Cloud Hosting,5200,6100,AWS,Variable,900,0.173077
14,2024-01-03,Product,Cloud Hosting,5000,5750,AWS,Variable,750,0.150000
8,2024-01-02,Product,Cloud Hosting,4700,5350,AWS,Variable,650,0.138298
35,2024-01-06,G&A,Professional Fees,4200,4800,Deloitte,Fixed,600,0.142857
17,2024-01-03,G&A,Professional Fees,4000,4600,Deloitte,Fixed,600,0.150000


In [5]:
expense_variance["Variance_Flag"] = np.where(
    expense_variance["Expense_Variance_%"] >= 0.10,
    "High Overrun",
    np.where(expense_variance["Expense_Variance_%"] > 0, "Moderate Overrun", "Within Budget")
)

expense_variance[["Month", "Department", "Expense_Category", "Vendor", "Budget_Expense", "Actual_Expense", "Expense_Variance_%", "Variance_Flag"]]

,Month,Department,Expense_Category,Vendor,Budget_Expense,Actual_Expense,Expense_Variance_%,Variance_Flag
0,2024-01-01,Sales,Payroll,Internal Payroll,12000,12000,0.000000,Within Budget
1,2024-01-01,Marketing,Marketing Spend,Google Ads,8000,9200,0.150000,High Overrun
2,2024-01-01,Product,Cloud Hosting,AWS,4500,5100,0.133333,High Overrun
3,2024-01-01,Customer Support,Customer Support Tools,Zendesk,2500,2400,-0.040000,Within Budget
4,2024-01-01,Operations,Software,Salesforce,3000,3150,0.050000,Moderate Overrun
5,2024-01-01,G&A,Rent,WeWork,6000,6000,0.000000,Within Budget
6,2024-01-02,Sales,Payroll,Internal Payroll,12000,12200,0.016667,Moderate Overrun
7,2024-01-02,Marketing,Marketing Spend,Google Ads,8500,8700,0.023529,Moderate Overrun
8,2024-01-02,Product,Cloud Hosting,AWS,4700,5350,0.138298,High Overrun
9,2024-01-02,Customer Support,Customer Support Tools,Zendesk,2500,2550,0.020000,Moderate Overrun


In [6]:
kpi_risk = kpi.copy()

kpi_risk["Churn_Risk"] = np.where(
    kpi_risk["Churn_Rate"] >= 0.08,
    "High Churn Risk",
    np.where(kpi_risk["Churn_Rate"] >= 0.04, "Moderate Churn Risk", "Low Churn Risk")
)

kpi_risk["CAC_Risk"] = np.where(
    kpi_risk["CAC"] >= 10000,
    "High CAC",
    np.where(kpi_risk["CAC"] >= 5000, "Moderate CAC", "Low CAC")
)

kpi_risk[["Month", "Active_Customers", "Churn_Rate", "CAC", "Net_Revenue_Retention", "Churn_Risk", "CAC_Risk"]]

,Month,Active_Customers,Churn_Rate,CAC,Net_Revenue_Retention,Churn_Risk,CAC_Risk
0,2024-01-01,2,0.0,4600,1.00,Low Churn Risk,Low CAC
1,2024-01-02,2,0.0,0,1.02,Low Churn Risk,Low CAC
2,2024-01-03,3,0.0,10300,1.03,Low Churn Risk,High CAC
3,2024-01-04,4,0.0,9800,1.04,Low Churn Risk,Moderate CAC
4,2024-01-05,5,0.0,11600,1.02,Low Churn Risk,High CAC
5,2024-01-06,5,0.2,10800,0.98,High Churn Risk,High CAC


In [7]:
latest_month_revenue = monthly_revenue.sort_values("Month").iloc[-1]["Total_Revenue"]

forecast = scenarios.copy()
forecast["Forecast_Revenue"] = latest_month_revenue * (1 + forecast["Customer_Growth_Rate"])

forecast[["Scenario", "Customer_Growth_Rate", "Churn_Rate", "CAC_Change", "Payroll_Inflation", "Forecast_Revenue"]]

,Scenario,Customer_Growth_Rate,Churn_Rate,CAC_Change,Payroll_Inflation,Forecast_Revenue
0,Base Case,0.08,0.04,0.05,0.04,10293.48
1,Upside Case,0.12,0.02,0.00,0.03,10674.72
2,Downside Case,0.04,0.08,0.12,0.06,9912.24


In [9]:
latest_kpi = kpi.sort_values("Month").iloc[-1]
latest_revenue = monthly_revenue.sort_values("Month").iloc[-1]["Total_Revenue"]
latest_expense = expenses[expenses["Month"] == expenses["Month"].max()]["Actual_Expense"].sum()

commentary = []

commentary.append(f"Latest monthly revenue was ${latest_revenue:,.2f}.")
commentary.append(f"Latest monthly operating expenses were ${latest_expense:,.2f}.")

if latest_kpi["Churn_Rate"] >= 0.08:
    commentary.append("Churn is a key risk area and should be monitored closely.")
else:
    commentary.append("Churn is currently within a manageable range.")

if latest_kpi["CAC"] >= 10000:
    commentary.append("Customer acquisition cost is high and may pressure profitability.")
else:
    commentary.append("Customer acquisition cost appears manageable.")

commentary.append("Management should focus on revenue quality, CAC control, churn reduction, and expense discipline.")

for line in commentary:
    print(line)

Latest monthly revenue was $9,531.00.
Latest monthly operating expenses were $43,750.00.
Churn is a key risk area and should be monitored closely.
Customer acquisition cost is high and may pressure profitability.
Management should focus on revenue quality, CAC control, churn reduction, and expense discipline.
